# 04 · Preprocessed Training Data

Run notebooks 01–03 first (they write `artifacts/`). The first run preprocesses every image into `cache/full_1024/`; later runs load from there.

In [1]:
# --- Setup: works both locally and on Kaggle ---
import os, sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    # Pull the shared src/ code from GitHub (requires Internet ON in notebook settings)
    if not Path("/kaggle/working/COS711-A2").exists():
        !git clone -q https://github.com/AryanMohanlall/COS711-A2.git /kaggle/working/COS711-A2
    ROOT = Path("/kaggle/working/COS711-A2")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT))

from src.data import get_data_dir
DATA_DIR = get_data_dir()
print("Kaggle:", IN_KAGGLE, "| data dir:", DATA_DIR)

Kaggle: False | data dir: /home/aryan/Documents/COS711/COS711-A2/data/MAGFiLO_1.0_Kaggle_2026


In [2]:
import pandas as pd

from src.data import load_annotations
from src.preprocess import build_cache
from src.dataset import FilamentDataset
from src.augment import AugConfig, NO_AUG

ART_DIR = ROOT / "artifacts"
VAL_FOLD = 0

images, anns = load_annotations(DATA_DIR)
excluded = pd.read_csv(ART_DIR / "excluded_annotations.csv")
anns = anns[~anns.id.isin(excluded.ann_id)].reset_index(drop=True)
geo_full = pd.read_csv(ART_DIR / "image_stats.csv").set_index("obs")[["cx", "cy", "radius"]]
folds = pd.read_csv(ART_DIR / "folds.csv")

paths = images.drop_duplicates("obs").set_index("obs").path
build_cache(paths.values, [tuple(geo_full.loc[o]) for o in paths.index])

data = dict(images=images, anns=anns, geo_full=geo_full)
train_ds = FilamentDataset(folds.loc[folds.fold != VAL_FOLD, "obs"], aug=AugConfig(), seed=0, **data)
val_ds = FilamentDataset(folds.loc[folds.fold == VAL_FOLD, "obs"], label="union", aug=NO_AUG, **data)

x, y, i = train_ds[0]   # x: preprocessed input, y: filament mask, both (1, 1024, 1024)
print(f"train {len(train_ds)} | val {len(val_ds)} |", x.shape, y.shape)

train 565 | val 142 | torch.Size([1, 1024, 1024]) torch.Size([1, 1024, 1024])
